In [5]:
!pip install transformers spacy sentence-transformers torch
!python -m spacy download en_core_web_sm


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 60.7 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
!pip uninstall -y transformers
!pip install transformers==4.36.2
!pip install torch sentencepiece
!python -m spacy download en_core_web_sm


Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.8/126.8 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.2/8.2 MB 63.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 25.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 67.0 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.4.0
    Uninstalling huggingface_hub-1.4.0:
      Successfully uninstalled huggingface_hub-1.4.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 88.8 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


#  AI - Contract Intelligence

In [ ]:
import re
import spacy
from transformers import pipeline

# Load Models


In [5]:
nlp = spacy.load("en_core_web_sm")

summarizer = pipeline(
    "summarization",
    model="facebook/bart-large-cnn",
    device=-1
)



# Sample Contract

In [6]:
contract_text = """
EMPLOYMENT AGREEMENT

This Employment Agreement is made on 01/01/2026.

The Company must pay salary of ₹50,000 per month.

The Employee shall maintain confidentiality of company data.

If there is a breach of contract, a penalty of ₹1,00,000 shall apply.

The Company may terminate employment with 30 days notice.
"""

#  Generic AI Summary

In [7]:
input_length = len(contract_text.split())
max_len = int(input_length * 0.6)

summary = summarizer(
    contract_text,
    max_length=max_len,
    min_length=30,
    do_sample=False
)[0]["summary_text"]

Your min_length=30 must be inferior than your max_length=28.


# CLEAN EXTRACTION

In [8]:
doc = nlp(contract_text)

# Proper Date Extraction
dates = []
for ent in doc.ents:
    if ent.label_ == "DATE":
        if any(char.isdigit() for char in ent.text):
            dates.append(ent.text)

dates = list(set(dates))

#  Proper Money Extraction
money_pattern = r"(₹\s?\d{1,3}(?:,\d{3})*(?:\.\d+)?)"
money = re.findall(money_pattern, contract_text)
money = list(set(money))


# Obligations
obligation_keywords = ["shall", "must", "required", "agree"]
obligations = []

for sent in doc.sents:
    sentence = sent.text.strip()
    if any(word in sentence.lower() for word in obligation_keywords):
        obligations.append(sentence)

obligations = list(set(obligations))

# Risk Clauses
risk_keywords = ["penalty", "breach", "liability", "fine"]
risks = []

for sent in doc.sents:
    sentence = sent.text.strip()
    if any(word in sentence.lower() for word in risk_keywords):
        risks.append(sentence)

risks = list(set(risks))

# Termination
termination = []

for sent in doc.sents:
    sentence = sent.text.strip()
    if "terminate" in sentence.lower():
        termination.append(sentence)

termination = list(set(termination))

# Improved Risk Score

risk_score = min(len(risks) * 30, 100)

#  Display Results

In [10]:
print("CONTRACT SUMMARY")
print(summary)
print("STRUCTURED SUMMARY")
print("Effective Dates:", dates)
print("Financial Terms:", money)
print("Obligations:", obligations)
print("Termination Clause:", termination)
print("Risk Clauses:", risks)
print("Overall Risk Score:", risk_score, "/100")

CONTRACT SUMMARY
Employment Agreement is made on 01/01/2026. Company must pay salary of ₹50,000 per
STRUCTURED SUMMARY
Effective Dates: ['30 days']
Financial Terms: ['₹50,000', '₹1']
Obligations: ['The Company must pay salary of ₹50,000 per month.', 'If there is a breach of contract, a penalty of ₹1,00,000 shall apply.', 'The Employee shall maintain confidentiality of company data.', 'EMPLOYMENT AGREEMENT\n\nThis Employment Agreement is made on 01/01/2026.']
Termination Clause: ['The Company may terminate employment with 30 days notice.']
Risk Clauses: ['If there is a breach of contract, a penalty of ₹1,00,000 shall apply.']
Overall Risk Score: 30 /100
